# Centralised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For CLSTM, there is one centralised model, so early stopping is checked once for the global training loop.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from clstm import (
    Datacollector,
    run_centralised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = "E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv" #'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 0
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'clstm'
args_truncated = None
args_SCALING_MODE = 'per_partition'  # 'global' or 'per_partition'
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'Scaling mode: {args_SCALING_MODE} | drop_boundary_gap: {args_DROP_BOUNDARY_GAP}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 20.06it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_9   lags_8   lags_7  \
0  12072.0  12162.0  12569.0  13238.0  ...  15177.0  15573.0  16281.0   
1  12162.0  12569.0  13238.0  14191.0  ...  15573.0  16281.0  16842.0   
2  12569.0  13238.0  14191.0  15213.0  ...  16281.0  16842.0  16503.0   
3  13238.0  14191.0  15213.0  15646.0  ...  16842.0  16503.0  15815.0   
4  14191.0  15213.0  15646.0  15653.0  ...  16503.0  15815.0  14745.0   

    lags_6   lags_5   lags_4   lags_3   lags_2   lags_1        y  
0  16842.0  16503.0  15815.0  14745.0

In [3]:
%%time
results = run_centralised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    scaling_mode=args_SCALING_MODE,
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
    checkpoint_path=f'fh{args_fh+1}_cen_lstm.pt',
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Stop epoch:', results['stop_epoch'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
pd.DataFrame(results['round_logs']).tail()

[Centralised] epoch 001/100 | avg_normalised_loss=0.004326 | avg_actual_loss=3459.832658 | delta=nan | stopped=False
[Centralised] epoch 002/100 | avg_normalised_loss=0.003199 | avg_actual_loss=2319.979293 | delta=0.00112666 | stopped=False
[Centralised] epoch 003/100 | avg_normalised_loss=0.003080 | avg_actual_loss=2249.576234 | delta=0.000118855 | stopped=False
[Centralised] epoch 004/100 | avg_normalised_loss=0.002567 | avg_actual_loss=1751.400137 | delta=0.00051323 | stopped=False
[Centralised] epoch 005/100 | avg_normalised_loss=0.002410 | avg_actual_loss=1629.338211 | delta=0.000157461 | stopped=False
[Centralised] epoch 006/100 | avg_normalised_loss=0.002261 | avg_actual_loss=1472.235139 | delta=0.000148154 | stopped=False
[Centralised] epoch 007/100 | avg_normalised_loss=0.002130 | avg_actual_loss=1311.522775 | delta=0.000131434 | stopped=False
[Centralised] epoch 008/100 | avg_normalised_loss=0.002293 | avg_actual_loss=1473.602401 | delta=0.00016313 | stopped=False
[Centralise

,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,stopped
23,24,0.001664,945.490892,0.000125,False
24,25,0.001792,1104.660041,0.000128,False
25,26,0.001671,948.023963,0.000121,False
26,27,0.001723,1005.792716,0.000052,False
27,28,0.001719,1018.449165,0.000004,True


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24835,6.667118,0.270386,0.277570
7,7,24835,16.508829,0.154720,0.163076
8,8,24835,12.619970,0.163457,0.170231
9,9,24835,12.536170,0.160938,0.166776
10,Overall,248350,19.578966,0.164056,0.169397


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24835,23.281356,0.944178,0.890582
7,7,24835,99.791777,0.935242,0.924299
8,8,24835,70.849026,0.917657,0.888018
9,9,24835,70.187424,0.901056,0.876588
10,Overall,248350,130.514211,0.920111,0.898427


In [10]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')
def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)

reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)

recon_results[0].head()

C:\Users\qifei\AppData\Local\Temp\ipykernel_6732\4248661303.py:55: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)


,unique_id,ds,y,clstm,clstm/BottomUp,clstm/MinTrace_method-mint_shrink,clstm/MinTrace_method-wls_var,clstm/MinTrace_method-ols,clstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15090.329387,15114.534479,15110.853025,15107.820718,15093.289379,15102.129844
1,Total,2014-07-01 00:00:00,13810.0,13808.771905,13806.775867,13806.996499,13807.146108,13808.451480,13807.609281
2,Total,2014-07-01 01:00:00,12979.0,12967.003014,12973.899199,12972.905356,12972.108063,12967.896970,12970.490754
3,Total,2014-07-01 02:00:00,12468.0,12424.607873,12433.434547,12432.382974,12431.629464,12425.954964,12429.575671
4,Total,2014-07-01 03:00:00,12226.0,12175.943341,12187.486709,12186.079372,12185.055067,12177.675497,12182.366785


In [11]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_cen'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='centralised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh1_cen_forecasts.csv
  per_series_metrics: fh1_cen_per_series_metrics.csv
  metrics_by_level: fh1_cen_metrics_by_level.csv
  overall_metrics: fh1_cen_overall_metrics.csv
  round_logs: fh1_cen_round_logs.csv
  timing: fh1_cen_timing.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,66.950977,0.115730,0.119120,1,centralised
1,1,top,bu,65.369367,0.112996,0.117306,1,centralised
2,1,top,mint_ols,66.576128,0.115082,0.118573,1,centralised
3,1,top,mint_shrinkage,65.385609,0.113024,0.117179,1,centralised
4,1,top,mint_var,65.471701,0.113172,0.117204,1,centralised
5,1,top,naive,527.787525,0.912318,0.895358,1,centralised
6,1,top,wls_structure,65.752271,0.113657,0.117479,1,centralised
7,2,middle,base,14.528953,0.174286,0.179128,6,centralised
8,2,middle,bu,14.493433,0.174150,0.179047,6,centralised
9,2,middle,mint_ols,14.689601,0.177349,0.180738,6,centralised


In [12]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,19.578966,0.164056,0.169397,10,centralised
1,bu,19.399494,0.163701,0.169167,10,centralised
2,mint_ols,19.651851,0.165961,0.170300,10,centralised
3,mint_shrinkage,19.392718,0.163547,0.168925,10,centralised
4,mint_var,19.420656,0.163782,0.169065,10,centralised
5,naive,130.514212,0.920111,0.898427,10,centralised
6,wls_structure,19.473441,0.164063,0.168938,10,centralised


In [13]:
timing_df

,module,seconds
0,training_sec,1621.287810
1,prediction_sec,21.479052
2,total_sec,1654.761887
